# MATH1810 - Course Setup

## Introduction to Scientific Python

Run the cell below (click it, then press **Shift-Enter**).

It will ask permission to use your Google Drive, and will then put the
course notebooks into **My Drive / MATH1810 / math1810**.

You can run this notebook again at any time to get updated material.
Your own work is never overwritten: anything you have changed is kept in a
numbered backup folder, and the cell tells you exactly which files those are.


In [ ]:
"""MATH1810 course installer.

Downloads a published release of the course and installs it into the student's
Google Drive at MyDrive/MATH1810/math1810.

Design rules, in order of importance:

  1. A failed or interrupted update must never damage a working installation.
     Nothing in Drive is touched until the archive has been downloaded and
     every file in it verified against the checksums it ships with.
  2. The student's own work is never overwritten in place. The previous course
     folder is preserved as a numbered backup, and files the student had
     modified are listed by name so they know what is in there.
  3. Exercise state (the id that fixes their questions, and attempt counters)
     survives updates.
  4. No Git, no credentials, no access to a private repository.

This module is the single source of truth: tools/build_release.py embeds this
file into the onboarding notebook the students actually run, and the tests
import it directly.
"""

from __future__ import annotations

import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import urllib.error
import urllib.request
import zipfile

# Set by the build to the published release index. Until the public repository
# exists this stays a placeholder and callers must pass index_url or archive_url.
DEFAULT_INDEX_URL = "https://raw.githubusercontent.com/notulae/math1810/main/release.json"

COURSE_DIRNAME = "math1810"
MANIFEST_NAME = "MANIFEST.sha256"
INSTALLED_MANIFEST = ".manifest.sha256"
# Exercise state that lives inside the course folder and must survive an update.
CARRY_FORWARD = re.compile(r"^\.(a\d+_counter\.npy|b\d+_counter\.npy|q\d+_counter\.npy|ts1\.txt|dho_counter\.npy|drivendho_counter\.npy)$")

TIMEOUT = 60


class InstallError(RuntimeError):
    """Something went wrong early enough that Drive was left untouched."""


def _fetch(url: str, timeout: int = TIMEOUT) -> bytes:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            return response.read()
    except urllib.error.HTTPError as exc:
        raise InstallError(f"Download failed ({exc.code}) for {url}") from exc
    except Exception as exc:  # URLError, timeout, DNS, ...
        raise InstallError(
            f"Could not download {url}\n"
            f"({exc}).\n"
            "Check your internet connection and run this cell again. "
            "Your existing course folder has not been changed."
        ) from exc


def _sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _read_manifest(text: str) -> dict:
    """Parse 'sha256␠␠path' lines into {path: sha256}."""
    entries = {}
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        digest, _, path = line.partition("  ")
        if not path:
            raise InstallError("Malformed manifest line: " + line)
        entries[path.strip()] = digest.strip()
    return entries


def resolve_release(index_url: str) -> dict:
    """Read the release index and return its metadata.

    The index names an immutable release; it is the only moving part, so a
    release tag never has to be repointed.
    """
    raw = _fetch(index_url)
    try:
        index = json.loads(raw.decode("utf-8"))
    except Exception as exc:
        raise InstallError(f"Release index at {index_url} is not valid JSON") from exc
    for key in ("version", "url", "sha256"):
        if key not in index:
            raise InstallError(f"Release index is missing '{key}'")
    if not index["url"].startswith(("https://", "file://")):
        raise InstallError("Release index points at a non-HTTPS URL; refusing.")
    return index


def _verify_and_extract(archive: bytes, expected_sha: str | None, target: str) -> dict:
    """Verify the archive, extract to `target`, verify every extracted file."""
    if expected_sha and _sha256(archive) != expected_sha:
        raise InstallError(
            "The downloaded file does not match its published checksum.\n"
            "Nothing has been changed. Please try again; if it keeps happening, "
            "tell your lecturer."
        )
    try:
        zf = zipfile.ZipFile(io.BytesIO(archive))
    except zipfile.BadZipFile as exc:
        raise InstallError("The download is not a valid archive (incomplete?).") from exc

    names = zf.namelist()
    for name in names:
        # Refuse anything that would write outside the target directory.
        if name.startswith("/") or ".." in name.split("/"):
            raise InstallError(f"Archive contains an unsafe path: {name}")
    prefix = COURSE_DIRNAME + "/"
    if not any(n.startswith(prefix) for n in names):
        raise InstallError(f"Archive does not contain a '{COURSE_DIRNAME}/' folder.")
    manifest_name = prefix + MANIFEST_NAME
    if manifest_name not in names:
        raise InstallError("Archive has no checksum manifest; refusing to install it.")

    zf.extractall(target)
    course = os.path.join(target, COURSE_DIRNAME)
    manifest = _read_manifest(open(os.path.join(course, MANIFEST_NAME)).read())
    missing, bad = [], []
    for rel, digest in manifest.items():
        path = os.path.join(course, rel)
        if not os.path.exists(path):
            missing.append(rel)
        elif _sha256(open(path, "rb").read()) != digest:
            bad.append(rel)
    if missing or bad:
        raise InstallError(
            "The downloaded course is incomplete or damaged "
            f"(missing: {missing}, wrong checksum: {bad}). Nothing has been changed."
        )
    return manifest


def _next_backup(module_root: str) -> str:
    pattern = re.compile(re.escape(COURSE_DIRNAME) + r"_backup_(\d{3})$")
    used = [
        int(m.group(1))
        for name in os.listdir(module_root)
        if os.path.isdir(os.path.join(module_root, name)) and (m := pattern.fullmatch(name))
    ]
    return os.path.join(module_root, f"{COURSE_DIRNAME}_backup_{max(used) + 1 if used else 1:03d}")


def _modified_since_install(course_dir: str) -> list:
    """Files the student changed, according to the manifest we wrote last time."""
    recorded = os.path.join(course_dir, INSTALLED_MANIFEST)
    if not os.path.exists(recorded):
        return []
    manifest = _read_manifest(open(recorded).read())
    changed = []
    for rel, digest in manifest.items():
        path = os.path.join(course_dir, rel)
        if not os.path.exists(path):
            changed.append(rel + " (deleted)")
        elif _sha256(open(path, "rb").read()) != digest:
            changed.append(rel)
    return sorted(changed)


def _carry_forward_state(backup_dir: str, course_dir: str) -> list:
    carried = []
    if not backup_dir or not os.path.isdir(backup_dir):
        return carried
    for name in os.listdir(backup_dir):
        if CARRY_FORWARD.match(name):
            try:
                shutil.copy2(os.path.join(backup_dir, name), os.path.join(course_dir, name))
                carried.append(name)
            except OSError:
                pass
    return sorted(carried)


def ensure_state_dir(module_root: str) -> tuple:
    """Create MyDrive/MATH1810/state and the persistent exercise id if absent."""
    import random

    state = os.path.join(module_root, "state")
    os.makedirs(state, exist_ok=True)
    id_path = os.path.join(state, "id.txt")
    if os.path.exists(id_path):
        try:
            value = int(open(id_path).read().strip())
            if 100000 <= value <= 999999:
                return state, value, False
        except (ValueError, OSError):
            pass
    value = random.randint(100000, 999999)
    with open(id_path, "w") as fh:
        fh.write(str(value))
    return state, value, True


def install(drive_root="/content/drive/MyDrive", index_url=None, archive_url=None,
            progress=print) -> dict:
    """Install or update the course. Returns a report dict."""
    index = None
    if archive_url is None:
        index_url = index_url or DEFAULT_INDEX_URL
        if index_url.startswith("@@"):
            raise InstallError(
                "This onboarding notebook has no release address configured. "
                "Please tell your lecturer."
            )
        progress("Looking up the current release...")
        index = resolve_release(index_url)
        archive_url, expected_sha, version = index["url"], index["sha256"], index["version"]
    else:
        expected_sha, version = None, "(direct download)"

    progress(f"Downloading course materials {version}...")
    archive = _fetch(archive_url)

    module_root = os.path.join(drive_root, "MATH1810")
    course_dir = os.path.join(module_root, COURSE_DIRNAME)

    with tempfile.TemporaryDirectory() as staging:
        progress("Checking the download...")
        manifest = _verify_and_extract(archive, expected_sha, staging)
        fresh = os.path.join(staging, COURSE_DIRNAME)

        # Everything below this line touches Drive; everything above could fail
        # safely. Keep this section as short and as ordinary as possible.
        os.makedirs(module_root, exist_ok=True)
        modified, backup = [], None
        if os.path.isdir(course_dir):
            modified = _modified_since_install(course_dir)
            backup = _next_backup(module_root)
            progress(f"Keeping your previous folder as {os.path.basename(backup)}...")
            os.rename(course_dir, backup)  # atomic, unlike copy-then-delete
        progress("Installing...")
        shutil.copytree(fresh, course_dir)

    with open(os.path.join(course_dir, INSTALLED_MANIFEST), "w") as fh:
        for rel in sorted(manifest):
            fh.write(f"{manifest[rel]}  {rel}\n")

    carried = _carry_forward_state(backup, course_dir)
    state_dir, exercise_id, id_created = ensure_state_dir(module_root)

    report = {
        "version": version,
        "course_dir": course_dir,
        "backup_dir": backup,
        "modified_by_student": modified,
        "carried_forward": carried,
        "exercise_id": exercise_id,
        "exercise_id_created": id_created,
        "file_count": len(manifest),
    }
    _print_report(report, progress)
    return report


def _print_report(report: dict, progress=print) -> None:
    progress("")
    progress("=" * 62)
    progress("  MATH1810 is installed")
    progress("=" * 62)
    progress(f"  Version : {report['version']} ({report['file_count']} files)")
    progress(f"  Folder  : My Drive / MATH1810 / {COURSE_DIRNAME}")
    if report["backup_dir"]:
        progress(f"  Previous version kept as: {os.path.basename(report['backup_dir'])}")
    if report["modified_by_student"]:
        progress("")
        progress("  You had changed these files. Your versions are in the backup")
        progress("  folder above - copy anything you want to keep back out of it:")
        for name in report["modified_by_student"]:
            progress(f"      {name}")
    if report["carried_forward"]:
        progress(f"  Exercise progress carried over ({len(report['carried_forward'])} files).")
    progress("")
    progress("  To start: open My Drive / MATH1810 / math1810 in Google Drive")
    progress("  and double-click Welcome.ipynb.")
    progress("=" * 62)



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

report = install()
